In [12]:
"""
Hungarian Riddle Benchmark - Claude Implementation (FIXED - Anti-Drift)
========================================================================
CRITICAL FIXES:
1. MUCH stricter prompts to prevent markdown headers (# Analysis, # ISMERETLEN)
2. Forces bracket-only format: [Answer]
3. Post-processing to strip ALL markdown and verbose preambles
4. Explicit "DO NOT" instructions for Claude's common drift patterns

Matches GPT's clean output format.
Developed with Claude AI assistance
"""

import pandas as pd
import litellm
import time
import os
import sys
import re

# =========================
# CONFIG
# =========================
MODEL_NAME = "claude-sonnet-4-5-20250929"
BENCHMARK_FILE = "100_HU_Riddles_Benchmark_Questions.tsv"

# Riddle range
ID_MIN = 1
ID_MAX = 100

OUTPUT_FILE = "claude_riddle_results_FIXED.tsv"

SLEEP_BETWEEN_CALLS = 1.5

# Console output options
SHOW_EXPECTED_ON_SCREEN = True
SHOW_RIDDLE_ON_SCREEN = False

# =========================
# AGGRESSIVE ANTI-DRIFT PROMPTS
# =========================

# STEP 1: Get SHORT answer ONLY (NO markdown, NO analysis)
ANSWER_SYSTEM_PROMPT = """You are a Hungarian cultural expert. You MUST follow these rules EXACTLY:

CRITICAL FORMAT RULES:
1. Output ONLY the answer in brackets: [Answer]
2. DO NOT write "# Analízis" or "# Elemzés" or ANY markdown headers
3. DO NOT write "Válasz:" or "A megfejtés:" or similar preambles
4. DO NOT explain your reasoning in this step
5. Maximum 1-4 words inside the brackets
6. If uncertain, write: [ISMERETLEN]

CORRECT examples:
[Unicum]
[Csaba és Gyula]
[Kürtőskalács]
[ISMERETLEN]

WRONG examples (DO NOT DO THIS):
# Analízis... ❌
# Kürtőskalács ❌
Válasz: Unicum ❌
[# ISMERETLEN] ❌
**Unicum** ❌

Output ONLY: [Answer]"""

REASONING_SYSTEM_PROMPT = """Te egy segítőkész magyar asszisztens vagy.

SZABÁLYOK:
1. CSAK magyarul válaszolj
2. 1-2 rövid mondat (maximum 3 mondat)
3. NE írj "Válasz:", "Indoklás:" címsort
4. NE használj markdown formázást
5. Magyarázd meg a MEGADOTT választ, NE változtasd meg

Csak a tiszta indoklást írd le, semmi mást."""

# =========================
# FEW-SHOT EXAMPLES
# =========================
FEW_SHOT_EXAMPLES = """
# Examples of CORRECT format (generic Hungarian culture, NOT from benchmark):

Riddle: Piros, gömbölyű, lédús.
Answer: [Paradicsom]

Riddle: Fehér, folyékony, tejtől van.
Answer: [Tej]

Riddle: Nagy magyar költő, "Az ember tragédiája" írója.
Answer: [Madách Imre]

Riddle: Ismert magyar desszert, mákos vagy diós töltelékkel.
Answer: [Bejgli]

Remember: ONLY output [Answer] in brackets, nothing else!
---
"""

# =========================
# POST-PROCESSING CLEANUP
# =========================
def aggressive_cleanup(text: str) -> str:
    """Remove ALL markdown, headers, preambles from Claude's response"""
    if not text:
        return "ISMERETLEN"
    
    # Remove markdown headers
    text = re.sub(r'^#+\s+.*$', '', text, flags=re.MULTILINE)
    
    # Remove common preambles
    text = re.sub(r'(?i)^\s*(Válasz|Answer|Megfejtés|A megfejtés|Elemzés|Analízis)\s*:\s*', '', text)
    
    # Remove "A válasz azért helyes..." type sentences
    text = re.sub(r'(?i)A válasz azért.*?mert.*?\.\s*', '', text)
    
    # Extract ONLY content inside brackets [...]
    bracket_match = re.search(r'\[(.*?)\]', text)
    if bracket_match:
        answer = bracket_match.group(1).strip()
        # Clean up any remaining markdown inside brackets
        answer = re.sub(r'[#*_`]', '', answer)
        return answer
    
    # If no brackets found, take first 1-4 words
    words = text.strip().split()[:4]
    clean = ' '.join(words)
    clean = re.sub(r'[#*_`\[\]]', '', clean)  # Remove markdown chars
    
    return clean.strip() or "ISMERETLEN"

def clean_reasoning(text: str) -> str:
    """Clean up reasoning text - remove preambles but keep content"""
    if not text:
        return ""
    
    # Remove "Válasz:" or "Indoklás:" prefixes
    text = re.sub(r'(?i)^\s*(Válasz|Indoklás|Magyarázat)\s*:\s*', '', text)
    
    # Remove markdown
    text = re.sub(r'[#*_`]', '', text)
    
    # Take only first 3 sentences max
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()][:3]
    
    return '. '.join(sentences) + '.' if sentences else ""

# =========================
# API CALL HELPER
# =========================
def call_claude(system_prompt: str, user_prompt: str, max_tokens: int = 100) -> str:
    """Call Claude via LiteLLM with retry logic"""
    params = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.0,
        "max_tokens": max_tokens,
        "api_key": os.environ.get("ANTHROPIC_API_KEY"),
    }
    
    for attempt in range(3):
        try:
            resp = litellm.completion(**params)
            return (resp.choices[0].message.content or "").strip()
        except Exception as e:
            if attempt < 2:
                time.sleep(2 * (attempt + 1))
            else:
                return f"API_ERROR: {str(e)[:200]}"
    
    return "API_ERROR: Max retries exhausted"

# =========================
# MAIN
# =========================
if __name__ == "__main__":
    print("🔑 Anthropic API Key:", "✓ Set" if os.environ.get("ANTHROPIC_API_KEY") else "✗ MISSING")
    if not os.environ.get("ANTHROPIC_API_KEY"):
        print("🛑 Missing ANTHROPIC_API_KEY in environment.")
        sys.exit(1)

    # Load benchmark
    try:
        df_all = pd.read_csv(BENCHMARK_FILE, sep="\t")
    except Exception as e:
        print(f"🛑 Failed to load TSV '{BENCHMARK_FILE}': {e}")
        sys.exit(1)

    # Filter by ID range
    df_all["ID_num"] = pd.to_numeric(df_all["ID"], errors="coerce")
    df = df_all[(df_all["ID_num"] >= ID_MIN) & (df_all["ID_num"] <= ID_MAX)].copy()
    df = df.sort_values("ID_num")

    if df.empty:
        print(f"🛑 No rows found for ID {ID_MIN}–{ID_MAX}.")
        sys.exit(1)

    print(f"\n=== CLAUDE FIXED BENCHMARK - IDs {ID_MIN}–{ID_MAX} ({len(df)} riddles) ===\n")

    results = []

    for _, row in df.iterrows():
        rid = int(row["ID_num"])
        topic = str(row.get("topic", "")).strip()
        riddle_text = str(row.get("riddle_text", "")).strip()
        expected = str(row.get("reference_answer", "")).strip()

        # Display expected (not saved to file)
        if SHOW_EXPECTED_ON_SCREEN:
            print(f"[ID {rid}] Expected: {expected}")
        else:
            print(f"[ID {rid}]")

        if SHOW_RIDDLE_ON_SCREEN:
            print(f"Riddle: {riddle_text}")

        # ===== STEP 1: Get SHORT ANSWER ONLY =====
        answer_user_prompt = (
            FEW_SHOT_EXAMPLES
            + f"\nNow answer this riddle:\n"
            + f"Topic: {topic}\n"
            + f"Riddle: {riddle_text}\n\n"
            + "Output ONLY: [Answer]"
        )
        
        raw_answer = call_claude(
            system_prompt=ANSWER_SYSTEM_PROMPT,
            user_prompt=answer_user_prompt,
            max_tokens=80
        )
        
        # Aggressive cleanup to extract clean answer
        final_answer = aggressive_cleanup(raw_answer)
        
        time.sleep(SLEEP_BETWEEN_CALLS)

        # ===== STEP 2: Get REASONING for the given answer =====
        reasoning_user_prompt = (
            f"Rejtvény: {riddle_text}\n"
            f"Az adott válasz: {final_answer}\n\n"
            f"Magyarázd meg magyarul 1-2 mondatban, miért illik ez a válasz erre a rejtvényre.\n"
            f"Csak az indoklást írd, semmi mást."
        )
        
        raw_reasoning = call_claude(
            system_prompt=REASONING_SYSTEM_PROMPT,
            user_prompt=reasoning_user_prompt,
            max_tokens=200
        )
        
        reasoning = clean_reasoning(raw_reasoning)

        # Display on screen (matching GPT format)
        print(f" -> [{final_answer}] | {reasoning}\n")

        # Save to results
        results.append({
            "ID": rid,
            "Topic": topic,
            "Riddle": riddle_text,
            "Final_Answer": final_answer,
            "Reasoning": reasoning
        })

        time.sleep(SLEEP_BETWEEN_CALLS)

    # Save to TSV
    out_df = pd.DataFrame(results)
    out_df.to_csv(OUTPUT_FILE, index=False, sep="\t")
    
    print(f"\n✅ Saved: {OUTPUT_FILE}")
    print(f"✅ Completed: {len(results)} riddles processed")
    print(f"\n📊 Quick Stats:")
    print(f"   - ISMERETLEN answers: {sum(1 for r in results if 'ISMERETLEN' in r['Final_Answer'].upper())}")
    print(f"   - Valid answers: {sum(1 for r in results if 'ISMERETLEN' not in r['Final_Answer'].upper())}")

🔑 Anthropic API Key: ✓ Set

=== CLAUDE FIXED BENCHMARK - IDs 1–100 (100 riddles) ===

[ID 1] Expected: Unicum
 -> [Unicum] | A rejtvény az Unicum keserű gyomorkeserű italra utal: a "vöröskeresztes gömb" a palack tetején lévő piros gömb alakú kupakra, a "nedve nem édes" a keserű ízre, a "hasadnak" pedig arra utal, hogy gyomor- és emésztési problémákra szokták inni. Az Unicum logója valóban egy piros gömb, és a ital hagyományosan gyomor- és emésztésjavító szerként ismert.

[ID 2] Expected: Csaba és Gyula
 -> [Pista és Pest] | A viccben a kisfiú azt mondja "Pista vagyok, Pestről jöttem, de nem tudom, mit akartam", ami egy klasszikus magyar poén a feledékenységről. A Pista keresztnév és a Pest városnév hasonló hangzása miatt keveredik össze a mondatban, ami vicces szójátékot eredményez.

[ID 3] Expected: Kürtőskalács
 -> [Kürtőskalács] | A kürtőskalácsot valóban forgó hengeren, nyílt láng felett sütik, ami hasonlít a kémény belsejéhez. A neve a "kürt" hangszerre utal, de a rejtvény kifejez